## Google Colab and data paths

Set `KDD_PROJECT_ROOT` if your project is not stored at `/content/drive/MyDrive/kdd_dropout_project`. The configured folder must contain `dataset/`, `shared/metadata/`, and `artifacts/`.


In [ ]:
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    pass

def discover_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").exists() and (candidate / "shared" / "metadata").exists():
            return candidate
    return start

default_root = (
    Path("/content/drive/MyDrive/kdd_dropout_project")
    if IN_COLAB else discover_project_root(Path.cwd())
)
PROJECT_ROOT = Path(os.environ.get("KDD_PROJECT_ROOT", default_root))
DATA_ROOT = PROJECT_ROOT / "dataset"
METADATA_ROOT = PROJECT_ROOT / "shared" / "metadata"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ARTIFACT_ROOT = ARTIFACT_ROOT / "baseline_bilstm"
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifacts: {MODEL_ARTIFACT_ROOT}")


In [2]:
import pandas as pd
import numpy as np
import torch
import os

print("Loading raw data...")
log_train = pd.read_csv(DATA_ROOT / "train/train/log_train.csv")
log_test = pd.read_csv(DATA_ROOT / "test/test/log_test.csv")
object_df = pd.read_csv(METADATA_ROOT / "object.csv")
truth_train = pd.read_csv(DATA_ROOT / "train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv(DATA_ROOT / "test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

# ==========================================
# 1. FILTER FIRST 2 WEEKS
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

# ==========================================
# 2. FIX: EXPLICIT MERGE (Resolves KeyError)
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)

# ==========================================
# 4. ENCODING
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat
    
    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)

# ==========================================
# 5. SLIDING WINDOWS
# ==========================================
MAX_LEN, STRIDE = 150, 75

def build_windows(df):
    src_all, evt_all, cat_all, gap_all, y_all, ids_all = [], [], [], [], [], []
    for _, row in df.iterrows():
        src, evt, cat, gaps = row["src"], row["evt"], row["cat"], row["gaps"]
        for i in range(0, len(src), STRIDE):
            s, e, c, g = src[i:i+MAX_LEN], evt[i:i+MAX_LEN], cat[i:i+MAX_LEN], gaps[i:i+MAX_LEN]
            if not s: continue
            
            pad_len = MAX_LEN - len(s)
            s += [0]*pad_len; e += [0]*pad_len; c += [0]*pad_len; g += [0]*pad_len
            
            src_all.append(s); evt_all.append(e); cat_all.append(c); gap_all.append(g)
            ids_all.append(row["student_id"])
            y_all.append(row["dropout"])
            
    return map(np.array, (src_all, evt_all, cat_all, gap_all, y_all, ids_all))

print("Building Windows...")
src_train, evt_train, cat_train, gap_train, y_train, _ = build_windows(train_df)
src_test, evt_test, cat_test, gap_test, y_test, test_ids = build_windows(test_df)

# ==========================================
# 6. SAVE PROCESSED DATA
# ==========================================
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
torch.save({
    "train": (src_train, evt_train, cat_train, gap_train, y_train),
    "test": (src_test, evt_test, cat_test, gap_test, y_test, test_ids),
    "vocabs": (source_vocab, event_vocab, category_vocab),
    "truth_test": truth_test
}, MODEL_ARTIFACT_ROOT / "processed_data.pt")

print(f"Success! Cleaned data saved to {MODEL_ARTIFACT_ROOT / 'processed_data.pt'}")

Loading raw data...
Building Windows...
Success! Cleaned data saved to data/processed/processed_data.pt
